In [ ]:
"""
Alpha Trading Research Platform

Notebook:
Sprint04_Strategy_Library

Purpose:
Run all four strategies (momentum, trend following, mean reversion,
breakout) through the same data/portfolio framework built in Sprint 3,
and compare them side by side.

Author:
Alison

Version:
0.5
"""

In [ ]:
# =====================================================
# ALPHA v0.5
# Sprint 4 - Strategy Library
# =====================================================

import matplotlib.pyplot as plt

from alpha.config import DEFAULT_CONFIG
from alpha.data import get_prices, get_monthly_prices
from alpha.portfolio import calculate_monthly_returns, build_portfolio

from alpha.strategies.momentum import calculate_monthly_momentum, select_top_stocks
from alpha.strategies.trend_following import select_trend_positions
from alpha.strategies.mean_reversion import select_oversold_stocks
from alpha.strategies.breakout import select_breakout_stocks

In [ ]:
# =====================================================
# CONFIGURATION
# =====================================================

config = DEFAULT_CONFIG
config

In [ ]:
# =====================================================
# DATA (shared across all four strategies)
# =====================================================

print("Downloading market data...")

prices = get_prices(config)
monthly_prices = get_monthly_prices(prices)
monthly_returns = calculate_monthly_returns(monthly_prices)

In [ ]:
# =====================================================
# RUN EACH STRATEGY THROUGH THE SAME PORTFOLIO FRAMEWORK
# =====================================================
# Positions are shifted by one month so we're never trading on a signal
# we wouldn't actually have had at the time.

monthly_momentum = calculate_monthly_momentum(monthly_prices, config)

position_sets = {
    "Momentum": select_top_stocks(monthly_momentum, config).shift(1),
    "Trend Following": select_trend_positions(monthly_prices, config).shift(1),
    "Mean Reversion": select_oversold_stocks(monthly_prices, config).shift(1),
    "Breakout": select_breakout_stocks(monthly_prices, config).shift(1),
}

results = {}

for name, positions in position_sets.items():
    returns, growth = build_portfolio(monthly_returns, positions)
    results[name] = {
        "returns": returns,
        "growth": growth,
        "avg_holdings": positions.sum(axis=1).mean(),
    }

In [ ]:
# =====================================================
# COMPARISON TABLE
# =====================================================
# Quick sanity check before Sprint 6 adds proper performance stats
# (CAGR, Sharpe, drawdown, etc.).

import pandas as pd

summary = pd.DataFrame({
    name: {
        "Final Growth": r["growth"].iloc[-1],
        "Avg Holdings/Month": r["avg_holdings"],
    }
    for name, r in results.items()
}).T

display(summary)

In [ ]:
# =====================================================
# COMPARISON CHART
# =====================================================

plt.figure(figsize=(12, 6))

for name, r in results.items():
    r["growth"].plot(label=name)

plt.title("Sprint 4 - Strategy Library Comparison")
plt.xlabel("Date")
plt.ylabel("Portfolio Growth")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# =====================================================
# NOTES
# =====================================================
# Momentum and trend following pick a stock the same conceptual way
# (recent price strength) but momentum ranks stocks against each other
# and always holds the top N, while trend following is a per-stock
# yes/no filter - so it can hold anywhere from zero to all of them in
# a given month. Same goes for mean reversion and breakout: they're
# filters, not rankings.
#
# That variable holding count is worth flagging for later - equal
# weighting across a variable number of positions changes how
# concentrated the portfolio's risk is month to month. Sprint 10
# (Portfolio Management) is where position sizing and portfolio-level
# risk get handled properly - worth revisiting these strategies then,
# especially against the 1%-per-trade risk rule.